# 023 checks

# Imports

In [ ]:
import json
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.stats import kstest
from tqdm.auto import tqdm

from uavloc.data.drone_frames import frame_place, load_frames
from uavloc.data.match_pairs import (PairBank, PairBankConfig, build_pair_bank, multidate_places, random_pose,
                                     random_prior, village_split)
from uavloc.data.multidate import load_date
from uavloc.data.satellite import GeoImage
from uavloc.features.factory import correlator_config
from uavloc.features.segmentation import SegmentationFeatures, load_segmenter
from uavloc.matching.benchmark import Draw, benchmark_draws
from uavloc.matching.correlator import MIN_WINDOW_STD, CorrelatorConfig, PositionCorrelator, class_ncc
from uavloc.matching.learned import (LearnedScorer, argmax_error_m, batched_ncc, build_head, heatmap_loss)
from uavloc.plotting.correlation import heatmap_grid
from uavloc.track import REPO_ROOT, experiment_dir, log_history, log_hparams, set_seed, tensorboard_writer

# Params

In [ ]:
EXP = "023-learned-correlation-head"
DEVICE = "mps"
SEED = 42

PLACES = ["bushcha-50p2422-25p8305", "birky-49p9105-23p9192"]
FRAMES_DATASET = "village-crossdate-frames"
REFERENCE_RUN = "007.1"
MAP_DATE = "newest"
NOISE_M = [5.0, 10.0, 15.0]
N_DIRECTIONS = 8
BENCH_SEED = 42
CLASS_WEIGHTS = (0.0, 1.0, 1.0, 0.5, 0.0, 0.5)

EXCLUDE_PLACES = ["birky-49p9105-23p9192", "bushcha-50p2422-25p8305", "dubrova-49p5175-24p0765"]
N_VAL_VILLAGES = 10
SPLIT_SEED = 42
DRONE_DATE = "current"
ALTITUDE_M = 40.0
FOV_W_DEG = 60.0
MARGIN_M = 80.0
SEARCH_SCALE = 1.7
MATCH_GSD_M = 0.385
SAT_FEATURE_CROP_PCT = 0.03
SEG_CHANNELS = "all"
SEG_CHECKPOINT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
N_TRAIN_PAIRS = 20_000
N_VAL_PAIRS = 2_000
BANK_SEED = 42

GEOMETRY_PAIRS = 300
MIN_TEXTURE_STD = 0.03            # gray query std (0-1) below which a same-date crop has nothing to lock on
MAX_CELL_ERROR = 1.0              # argmax within one cell of the truth, per axis
GEOMETRY_PASS_SHARE = 0.99
KS_MIN_P = 0.01
OVERFIT_PAIRS = 64                 # drawn from covered pairs only (amendment 2026-10-30)
OVERFIT_STEPS = 400
OVERFIT_BATCH = 32
OVERFIT_LR = 3e-3
OVERFIT_LOG_EVERY_STEPS = 50
OVERFIT_MAX_ERROR_M = 0.5
TARGET_SIGMA_CELLS = 1.0
ENCODER_CHANNELS = (32, 32, 16)
REFINE_CHANNELS = 16

# Data

In [ ]:
set_seed(SEED)
checks = {}

In [ ]:
train_villages, val_villages = village_split(EXCLUDE_PLACES, N_VAL_VILLAGES, SPLIT_SEED)
checks["split"] = {"train": len(train_villages), "val": len(val_villages),
                   "disjoint": not set(train_villages) & set(val_villages),
                   "excluded_absent": not set(EXCLUDE_PLACES) & set(train_villages + val_villages),
                   "covers_all_others": len(train_villages) + len(val_villages) + len(EXCLUDE_PLACES) == len(multidate_places())}
checks["split"]

In [ ]:
model = load_segmenter(REPO_ROOT / SEG_CHECKPOINT, DEVICE, "lraspp")
PAIR_CONFIG = PairBankConfig(DRONE_DATE, ALTITUDE_M, FOV_W_DEG, MARGIN_M, SEARCH_SCALE, MATCH_GSD_M,
                             SAT_FEATURE_CROP_PCT, SEG_CHANNELS, SEG_CHECKPOINT)
banks = {"train": build_pair_bank(train_villages, N_TRAIN_PAIRS, PAIR_CONFIG, model, DEVICE, BANK_SEED),
         "val": build_pair_bank(val_villages, N_VAL_PAIRS, PAIR_CONFIG, model, DEVICE, BANK_SEED)}
{split: (len(bank), bank.meta["surface_hw"], bank.cell_m) for split, bank in banks.items()}

# Offsets

In [ ]:
def uniformity(bank: PairBank) -> dict[str, float]:
    """KS p-value of the true row and column against uniform over the surface cells."""
    rows, cols = bank.meta["surface_hw"]
    return {"p_row": float(kstest(bank.pairs["cell_row"] / (rows - 1), "uniform").pvalue),
            "p_col": float(kstest(bank.pairs["cell_col"] / (cols - 1), "uniform").pvalue)}

In [ ]:
checks["offsets_uniform"] = {split: uniformity(bank) for split, bank in banks.items()}
checks["pairs_per_place"] = {split: bank.pairs["place"].value_counts().agg(["min", "max"]).to_dict()
                             for split, bank in banks.items()}
checks["map_dates"] = {split: bank.pairs["map_date"].value_counts().to_dict() for split, bank in banks.items()}
checks["offsets_uniform"], checks["pairs_per_place"]

# Parity with 007.1

In [ ]:
def parity_rows(correlator: PositionCorrelator, draws: list[Draw], scorer: LearnedScorer) -> list[dict]:
    """Per draw: batched FFT NCC vs `class_ncc`, and the weights head at the hand weights vs weighted NCC."""
    rows = []
    for draw in draws:
        if draw.surface is None:
            continue
        frame = draw.frame
        drone, sat, _, _ = correlator.prepare(correlator.drone_map(frame), draw.prior, frame.yaw_deg,
                                              correlator.patch_size_m(frame))
        batched = batched_ncc(drone, sat, MIN_WINDOW_STD)[0]
        direct = torch.stack([class_ncc(drone[0, c].float().cpu(), sat[0, c].float().cpu()) for c in range(drone.shape[1])])
        reference = draw.surface.heatmap
        covered = draw.surface.coverage > 0 and np.nanmax(reference) > np.nanmin(reference)
        rows.append({"frame": frame.name, "noise_m": draw.noise_m, "covered": covered,
                     "max_ncc_diff": float((batched - direct).abs().max()),
                     "same_argmax": bool(np.argmax(scorer(drone, sat, None)) == np.argmax(reference))})
    return rows

In [ ]:
frames = load_frames(FRAMES_DATASET)
weights_scorer = LearnedScorer(build_head("weights", CLASS_WEIGHTS, ENCODER_CHANNELS, REFINE_CHANNELS), "cpu")
parity = []
for place in tqdm(PLACES, desc="places"):
    features = SegmentationFeatures(model, model, load_date(place, MAP_DATE), SEG_CHANNELS, MATCH_GSD_M, MATCH_GSD_M, DEVICE)
    correlator = PositionCorrelator(features, correlator_config({"search_scale": SEARCH_SCALE, "fov_w_deg": FOV_W_DEG,
                                                                 "sat_feature_crop_pct": SAT_FEATURE_CROP_PCT,
                                                                 "drone_feature_crop_pct": 0.0, "matching": "weighted-ncc",
                                                                 "class_weights": CLASS_WEIGHTS}))
    members = [f for f in frames if frame_place(f) == place]
    draws = [d for noise_m in NOISE_M for d in benchmark_draws(correlator, members, noise_m, N_DIRECTIONS, BENCH_SEED)]
    parity += parity_rows(correlator, draws, weights_scorer)
parity = pd.DataFrame(parity)

In [ ]:
covered = parity[parity["covered"]]
checks["parity"] = {"draws": len(parity), "covered_draws": len(covered),
                    "max_ncc_diff": float(parity["max_ncc_diff"].max()),
                    "weights_head_same_argmax_share_covered": float(covered["same_argmax"].mean())}
checks["parity"]

# Geometry

In [ ]:
class GrayFeatures:
    """One gray channel of an RGB raster on both sides: a geometry check that involves no model."""

    def __init__(self, image: GeoImage, gsd_m: float) -> None:
        self.image, self.gsd_m = image, gsd_m

    def patch(self, center, yaw_deg: float, width_m: float, height_m: float) -> torch.Tensor:
        rgb = self.image.crop_rotated(center, yaw_deg, width_m, height_m, out_mpp=self.gsd_m)
        return torch.from_numpy(rgb.mean(2, dtype=np.float32) / 255)[None, None]

    def satellite_patch(self, center, yaw_deg: float, width_m: float, height_m: float) -> torch.Tensor:
        return self.patch(center, yaw_deg, width_m, height_m)

In [ ]:
def geometry_rows(places: list[str], n: int, seed: int) -> pd.DataFrame:
    """Same-date gray query vs gray patch around a prior drawn like the bank's: argmax cell vs the true cell."""
    rng = np.random.default_rng(seed)
    rows, template = [], None
    for place in tqdm(rng.choice(places, size=n), desc="geometry"):
        features = GrayFeatures(load_date(place, DRONE_DATE), MATCH_GSD_M)
        correlator = PositionCorrelator(features, PAIR_CONFIG.correlator_config())
        truth, yaw_deg = random_pose(rng, features.image, MARGIN_M)
        query = features.patch(truth, yaw_deg, *PAIR_CONFIG.footprint_m)
        if template is None:
            template = correlator.prepare(query, truth, yaw_deg, PAIR_CONFIG.patch_size_m)[2]
        prior = random_prior(rng, truth, yaw_deg, template)
        drone, sat, surface, _ = correlator.prepare(query, prior, yaw_deg, PAIR_CONFIG.patch_size_m)
        true_cell = surface.cell_of_offset(PositionCorrelator.patch_offset_m(prior, truth, yaw_deg))
        best = np.unravel_index(batched_ncc(drone, sat, 0.0)[0, 0].argmax().item(), template.heatmap.shape)
        rows.append({"place": place, "texture_std": float(query.std()),
                     "cell_error": max(abs(best[0] - true_cell[0]), abs(best[1] - true_cell[1]))})
    return pd.DataFrame(rows)

In [ ]:
geometry = geometry_rows(train_villages, GEOMETRY_PAIRS, SEED)
textured = geometry[geometry["texture_std"] >= MIN_TEXTURE_STD]
checks["geometry"] = {"pairs": len(geometry), "textured": len(textured),
                      "within_one_cell_share": float((textured["cell_error"] <= MAX_CELL_ERROR).mean()),
                      "pass": bool((textured["cell_error"] <= MAX_CELL_ERROR).mean() >= GEOMETRY_PASS_SHARE)}
checks["geometry"]

# Overfit

In [ ]:
def covered_indices(bank: PairBank, n: int) -> np.ndarray:
    """First `n` pairs whose query holds a class with nonzero hand weight: blind queries carry no position to memorise."""
    presence = np.asarray(bank.arrays["presence"])
    covered = np.flatnonzero(presence[:, np.array(CLASS_WEIGHTS) > 0].any(1))
    return covered[:n]


In [ ]:
def overfit(arm: str, bank: PairBank, indices: np.ndarray) -> pd.DataFrame:
    """Adam on the same `indices` only; loss and mean argmax error every `OVERFIT_LOG_EVERY_STEPS`, to TensorBoard too."""
    writer = tensorboard_writer(f"{EXP}/overfit-{arm}")
    head = build_head(arm, CLASS_WEIGHTS, ENCODER_CHANNELS, REFINE_CHANNELS).to(DEVICE)
    opt = torch.optim.Adam(head.parameters(), lr=OVERFIT_LR)
    rng = np.random.default_rng(SEED)
    rows = []
    for step in tqdm(range(OVERFIT_STEPS + 1), desc=f"overfit {arm}"):
        if step % OVERFIT_LOG_EVERY_STEPS == 0:
            head.eval()
            with torch.no_grad():
                batch = bank.batch(indices)
                logits = head(batch["query"], batch["satellite"], batch["ncc"], batch["presence"])
                rows.append({"step": step, "loss": heatmap_loss(logits, batch["cell"], TARGET_SIGMA_CELLS).item(),
                             "error_m": float(argmax_error_m(logits, batch["cell"], bank.cell_m).mean())})
            log_history(writer, {"step": step, "train_loss": rows[-1]["loss"], "train_error_m": rows[-1]["error_m"]})
        head.train()
        batch = bank.batch(np.sort(rng.choice(indices, OVERFIT_BATCH, replace=False)))
        loss = heatmap_loss(head(batch["query"], batch["satellite"], batch["ncc"], batch["presence"]), batch["cell"],
                            TARGET_SIGMA_CELLS)
        opt.zero_grad(); loss.backward(); opt.step()
    log_hparams(writer, {"arm": arm, "pairs": len(indices), "steps": OVERFIT_STEPS, "batch": OVERFIT_BATCH,
                         "lr": OVERFIT_LR, "encoder_channels": ENCODER_CHANNELS, "refine_channels": REFINE_CHANNELS},
                {"loss_end": rows[-1]["loss"], "error_m_end": rows[-1]["error_m"]})
    return pd.DataFrame(rows)

In [ ]:
overfit_indices = covered_indices(banks["val"], OVERFIT_PAIRS)
overfit_curves = {arm: overfit(arm, banks["val"], overfit_indices) for arm in ("weights", "head")}
checks["overfit"] = {arm: {"loss_start": c["loss"].iloc[0], "loss_end": c["loss"].iloc[-1],
                           "error_m_start": c["error_m"].iloc[0], "error_m_end": c["error_m"].iloc[-1]}
                     for arm, c in overfit_curves.items()}
checks["overfit"]["head"]["pass"] = bool(checks["overfit"]["head"]["error_m_end"] < OVERFIT_MAX_ERROR_M)
checks["overfit"]["weights"]["pass"] = bool(checks["overfit"]["weights"]["loss_end"] < checks["overfit"]["weights"]["loss_start"])
checks["overfit"]

# Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, axis in zip(axes, ("cell_row", "cell_col")):
    for split, bank in banks.items():
        ax.hist(bank.pairs[axis], bins=40, histtype="step", density=True, label=split)
    ax.set_xlabel(axis); ax.legend()
fig.tight_layout()

In [ ]:
sample = banks["val"].batch(np.arange(8))
plt.figure(figsize=(12, 4)); plt.imshow(heatmap_grid(sample["ncc"][:, 1].numpy(), sample["cell"].numpy())); plt.axis("off")
plt.title("building NCC of 8 val pairs: white circle truth, red cross argmax")

# Save

In [ ]:
artifacts = experiment_dir(EXP) / "artifacts"
artifacts.mkdir(exist_ok=True)
(artifacts / "checks.json").write_text(json.dumps(checks, indent=1, default=float))
checks